In [49]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [50]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2

In [51]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [52]:
target_col = 'Output'

# First set of features (Model explainability with log regression)
feature_cols_1 = ['TCD', 'PC', 'CN0', 'EC', 'PRN', 'LC', 'DO', 'PD' ,'CP', 'PIP']
X1 = df[feature_cols_1]
y = df[target_col]

# Second set of features (From Xgboost model with all features)
feature_cols_2 = ['TOW', 'PD', 'PRN', 'RX', 'TCD', 'DO', 'CP', 'CN0', 'PC', 'EC']
X2 = df[feature_cols_2]

# Split both sets with the same random state
X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)
X2_train, X2_test, _, _ = train_test_split(
    X2, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)  # y_train/y_test already made above

# Scale both sets
scaler1 = StandardScaler()
X1_train_scaled = scaler1.fit_transform(X1_train)
X1_test_scaled = scaler1.transform(X1_test)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

In [53]:
# Train and evaluate first model
model1 = XGBClassifier(
    objective='multi:softprob',
    num_class= NUM_CLASSES,
    eval_metric='mae',
    random_state=RANDOM_STATE,
)

model1.fit(X1_train_scaled, y_train)
y1_pred = model1.predict(X1_test_scaled)
accuracy1 = accuracy_score(y_test, y1_pred)
print(f"Model 1 (Model explainability) Accuracy: {accuracy1 * 100:.2f}%")
print("Classification report (Model 1):")
print(classification_report(y_test, y1_pred, zero_division=0))

Model 1 (Model explainability) Accuracy: 93.14%
Classification report (Model 1):
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    318260
           1       0.81      0.81      0.81     29166
           2       0.73      0.70      0.72     35386
           3       0.95      0.94      0.94     25612

    accuracy                           0.93    408424
   macro avg       0.86      0.86      0.86    408424
weighted avg       0.93      0.93      0.93    408424



In [54]:
# Train and evaluate second model
model2 = XGBClassifier(
    objective='multi:softprob',
    num_class= NUM_CLASSES,
    eval_metric='mae',
    random_state=RANDOM_STATE,
)
model2.fit(X2_train_scaled, y_train)
y2_pred = model2.predict(X2_test_scaled)
accuracy2 = accuracy_score(y_test, y2_pred)
print(f"Model 2 (Xgboost features) Accuracy: {accuracy2 * 100:.2f}%")
print("Classification report (Model 2):")
print(classification_report(y_test, y2_pred, zero_division=0))

Model 2 (Xgboost features) Accuracy: 93.32%
Classification report (Model 2):
              precision    recall  f1-score   support

           0       0.96      0.97      0.97    318260
           1       0.82      0.82      0.82     29166
           2       0.73      0.71      0.72     35386
           3       0.95      0.94      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.87      0.86      0.86    408424
weighted avg       0.93      0.93      0.93    408424

